In [ ]:
#SETUP

from pathlib import Path
from datetime import date
import os
import time

import pandas as pd
import requests
from dotenv import load_dotenv
from fredapi import Fred 

load_dotenv(Path("..")/".env")
FRED_KEY = os.getenv("FRED_API_KEY")
BEA_KEY = os.getenv("BEA_API_KEY")

RAW = Path("../data/raw")
RAW.mkdir(parents=True, exist_ok=True)

START = "1989-01-01"
TODAY = date.today().isoformat()

In [ ]:
fred = Fred(api_key=FRED_KEY)

fred_series = {
    "unrate" : "UNRATE",       #unemployment rate %
    "cpi" : "CPIAUCSL",        #consumer price index(used for inflation)
    "pce_price" : "PCEPI",     #personal consumption expenditures price index
    "saving_rate" : "PSAVERT", #personal saving rate %
    "sentiment" : "UMCSENT",  #University of Michigan consumer sentiment
    "usrec" : "USREC",         #NBER recession indicator: 1 = recession month, 0 = not in recession
    }

fred_data = {}
for name, series_id in fred_series.items():
    fred_data[name] = fred.get_series(
        series_id, observation_start=START
    )
    print(f"{name:12s} {series_id:10s} {len(fred_data[name])} obs")

fred_macro = pd.DataFrame(fred_data)
fred_macro.index.name = "date"
fred_macro.to_csv(RAW / f"fred_macro_{TODAY}.csv")
fred_macro.tail()


In [ ]:
def get_bea_table(table_name, start_year=1989, end_year=None, chunk=5):
    end_year=end_year or date.today().year
    frames=[]

    all_years = list(range(start_year, end_year + 1))

    for i in range(0, len(all_years), chunk):
        batch = all_years[i:i + chunk]
        years_str = ",". join(str(y) for y in batch)

        params = {                                 
            "UserID": BEA_KEY,                    
            "method": "GetData",                   
            "datasetname": "NIUnderlyingDetail",   
            "TableName": table_name,               
            "Frequency": "M",                      
            "Year": years_str,                        
            "ResultFormat": "json",               
        }
        r = requests.get(                        
            "https://apps.bea.gov/api/data",       
            params=params,                         
            timeout=120,                           
        )
        r.raise_for_status()                      

        res = r.json()["BEAAPI"]                  
        if "Error" in res or "Error" in res.get("Results", {}):  
            raise RuntimeError(res)                

        frames.append(                            
            pd.DataFrame(res["Results"]["Data"])  
        )
        time.sleep(1)                              

    return pd.concat(frames, ignore_index=True)  
   

  Table 2.4.6U (real, chained dollars) is not downloaded because BEA does not
  provide it for the full period from 1989. Real series are computed in
  preprocessing as nominal / price index × 100.

In [ ]:
bea_tables = {
    "U20405": "bea_2_4_5U_nominal", 
    "U20404": "bea_2_4_4U_prices",  
}

for table, name in bea_tables.items():
    df = get_bea_table(table)
    df.to_csv(RAW / f"{name}_{TODAY}.csv", index=False)
    print(f"{table}: {len(df):,} rows saved")


In [ ]:
nominal = pd.read_csv(RAW / f"bea_2_4_5U_nominal_{TODAY}.csv")  
print(nominal.columns.tolist())                                  

nominal[["LineNumber", "LineDescription"]].drop_duplicates().head(60)  

In [ ]:
lines = nominal[["LineNumber", "LineDescription", "SeriesCode"]].drop_duplicates()

wanted = (
    "^(Personal consumption expenditures|Goods|Durable goods|Nondurable goods"
    "|Food and beverages purchased for off-premises|Clothing and footwear"
    "|Services|Housing and utilities|Recreational goods and vehicles"
    "|Recreation services|Food services and accommodations)"
)

lines[lines["LineDescription"].str.contains(wanted, case=False, regex=True)]